In [4]:
# Block 1: Setup, Imports, and Data Loading
# ==============================================================================
import warnings
import time
import pandas as pd
import numpy as np
import gc

# Model Imports
import xgboost as xgb
import lightgbm as lgb
import catboost as cb

# Sklearn Imports
from sklearn.model_selection import StratifiedKFold
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import mean_squared_error
from sklearn.cluster import MiniBatchKMeans

# Ignore warnings
warnings.filterwarnings('ignore')

print("Libraries imported successfully.")

# --- Load Data ---
try:
    # We drop the low-variance columns they identified right away
    drop_cols=['id', 'golf', 'view_rainier', 'view_skyline',
               'view_lakesamm', 'view_otherwater', 'view_other']
    DATA_PATH = './'
    df_train = pd.read_csv(DATA_PATH + 'dataset.csv').drop(columns=drop_cols)
    df_test = pd.read_csv(DATA_PATH + 'test.csv').drop(columns=drop_cols)
    print("Raw data loaded successfully.")
except FileNotFoundError:
    print("ERROR: Could not find 'dataset.csv' or 'test.csv'.")
    exit()

# --- Global Constants & Target Variable ---
N_SPLITS = 5
RANDOM_STATE = 42
COMPETITION_ALPHA = 0.1

y_true = df_train['sale_price'].copy()
# We keep sale_price in df_train for feature engineering
print("Setup complete.")

# --- Global Model Parameter Dictionaries ---
print("\nDefining global model parameters...")

# These will be updated by the Optuna tuning block later
xgb_params = {
    'objective': 'reg:squarederror', 
    'eval_metric': 'rmse', 
    'eta': 0.02, # placeholder
    'max_depth': 7, # placeholder
    'subsample': 0.8, # placeholder
    'colsample_bytree': 0.7, # placeholder
    'random_state': RANDOM_STATE, 
    'n_jobs': -1
}

lgb_params = {
    'objective': 'regression_l1', 
    'metric': 'rmse', 
    'n_estimators': 2000, 
    'learning_rate': 0.01, # placeholder
    'feature_fraction': 0.8, # placeholder for colsample_bytree
    'bagging_fraction': 0.8, # placeholder for subsample
    'bagging_freq': 1, 
    'lambda_l1': 0.1, # placeholder for reg_alpha
    'lambda_l2': 0.1, # placeholder for reg_lambda
    'num_leaves': 31, # placeholder
    'verbose': -1, 
    'n_jobs': -1, 
    'seed': RANDOM_STATE
}

cb_params = {
    'iterations': 2000, 
    'learning_rate': 0.02, # placeholder
    'depth': 7, # placeholder
    'loss_function': 'RMSE', 
    'eval_metric': 'RMSE', 
    'random_seed': RANDOM_STATE, 
    'verbose': 0
    # 'cat_features' will be added inside the training loop
}

print("Global parameters defined.")

Libraries imported successfully.
Raw data loaded successfully.
Setup complete.

Defining global model parameters...
Global parameters defined.


In [5]:
# Block 2: Advanced Feature Engineering on Steroids
# ==============================================================================
print("--- Starting Block 2: Feature Engineering on Steroids ---")

def feature_engineer_steroids(df, train_df=None):
    """Creates an even richer set of features."""

    # --- A) Brute-Force Numerical Interactions ---
    NUMS = ['area', 'land_val', 'imp_val', 'sqft_lot', 'sqft', 'sqft_1', 'grade', 'year_built']
    for i in range(len(NUMS)):
        for j in range(i + 1, len(NUMS)):
            df[f'{NUMS[i]}_x_{NUMS[j]}'] = df[NUMS[i]] * df[NUMS[j]]

    # --- B) Date Features ---
    df['sale_date'] = pd.to_datetime(df['sale_date'])
    df['year'] = df['sale_date'].dt.year
    df['month'] = df['sale_date'].dt.month
    df['weekofyear'] = df['sale_date'].dt.isocalendar().week.astype(int)
    df['dayofyear'] = df['sale_date'].dt.dayofyear
    df['quarter'] = df['sale_date'].dt.quarter
    
    df['year_diff'] = df['year'] - df['year_built']
    df['year_reno_diff'] = df['year'] - df['year_reno'].replace(0, np.nan)
    df['month_total'] = df['year'] * 12 + df['month'] # A continuous time feature

    # --- C) Geospatial Features ---
    coords = df[['latitude', 'longitude']].copy()
    kmeans = MiniBatchKMeans(n_clusters=20, random_state=RANDOM_STATE, batch_size=256, n_init=10) # Increased clusters
    df['geo_cluster'] = kmeans.fit_predict(coords)
    df['lat_lon'] = df['latitude'] * df['longitude']

    # --- D) Advanced Target Encoding (on geo_cluster) ---
    if train_df is not None:
        # Calculate mappings from the provided training dataframe
        geo_price_mean = train_df.groupby('geo_cluster')['sale_price'].mean()
        geo_price_std = train_df.groupby('geo_cluster')['sale_price'].std()
        
        # Apply mappings to the current dataframe
        df['geo_cluster_price_mean'] = df['geo_cluster'].map(geo_price_mean)
        df['geo_cluster_price_std'] = df['geo_cluster'].map(geo_price_std)
        
        # Fill any missing values with the global mean/std
        df['geo_cluster_price_mean'].fillna(train_df['sale_price'].mean(), inplace=True)
        df['geo_cluster_price_std'].fillna(train_df['sale_price'].std(), inplace=True)

    # --- E) Log transform ---
    for c in ['land_val_x_imp_val', 'land_val_x_sqft', 'imp_val_x_sqft']:
        if c in df.columns:
            df[c] = np.log1p(df[c])

    return df

# --- Re-run the data preparation with the new function ---
# Load original data again to ensure a clean slate
df_train = pd.read_csv(DATA_PATH + 'dataset.csv').drop(columns=drop_cols)
df_test = pd.read_csv(DATA_PATH + 'test.csv').drop(columns=drop_cols)

# Create geo_cluster on train_df first to pass it for target encoding
coords_train = df_train[['latitude', 'longitude']].copy()
kmeans_train = MiniBatchKMeans(n_clusters=20, random_state=RANDOM_STATE, batch_size=256, n_init=10)
df_train['geo_cluster'] = kmeans_train.fit_predict(coords_train)

# Apply the full feature engineering
df_train = feature_engineer_steroids(df_train, train_df=df_train.copy()) # Pass itself to learn mappings
df_test = feature_engineer_steroids(df_test, train_df=df_train.copy()) # Pass train_df to map to test

# Combine data for consistent final processing
train_ids = df_train.index
test_ids = df_test.index
df_train['is_train'] = 1
df_test['is_train'] = 0
all_data = pd.concat([df_train.drop('sale_price', axis=1), df_test], axis=0).reset_index(drop=True)


# Final Cleanup
cols_to_drop_final = ['sale_date']
all_data = all_data.drop(columns=cols_to_drop_final)

# Intelligent Imputation (same as before)
text_cols_impute = ['subdivision', 'zoning', 'city', 'sale_warning', 'join_status', 'submarket']
for col in all_data.columns:
    if col in text_cols_impute or all_data[col].dtype == 'object':
        all_data[col] = all_data[col].fillna('missing')
    else:
        all_data[col] = all_data[col].fillna(0)

# Separate back into X and X_test
X = all_data[all_data['is_train'] == 1].drop(columns=['is_train'])
X_test = all_data[all_data['is_train'] == 0].drop(columns=['is_train'])

# Restore original index
X.index = train_ids
X_test.index = test_ids

print(f"\nSynthesized FE complete. Total features: {X.shape[1]}")
gc.collect()

--- Starting Block 2: Feature Engineering on Steroids ---

Synthesized FE complete. Total features: 78


313

In [3]:
# Block 2.5: Hyperparameter Tuning with Optuna
# ==============================================================================
import optuna

print("\n--- Starting Block 2.5: Hyperparameter Tuning ---")

# We will tune on a single validation split for speed
# Using a simple train/test split here, not full CV
from sklearn.model_selection import train_test_split
X_train_opt, X_val_opt, y_train_opt, y_val_opt = train_test_split(X, y_true, test_size=0.2, random_state=RANDOM_STATE)

def objective(trial):
    """Define the objective function for Optuna to minimize."""
    
    # We will tune LightGBM as it's fast and powerful
    lgb_params_opt = {
        'objective': 'regression_l1',
        'metric': 'rmse',
        'random_state': RANDOM_STATE,
        'n_estimators': 1000, # Fixed for tuning speed
        'verbose': -1,
        'n_jobs': -1,
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.1, log=True),
        'num_leaves': trial.suggest_int('num_leaves', 20, 300),
        'max_depth': trial.suggest_int('max_depth', 3, 12),
        'subsample': trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 1.0),
        'reg_alpha': trial.suggest_float('reg_alpha', 1e-8, 10.0, log=True),
        'reg_lambda': trial.suggest_float('reg_lambda', 1e-8, 10.0, log=True),
        'min_child_samples': trial.suggest_int('min_child_samples', 5, 100),
    }

    # For tuning, we can drop the text columns temporarily as TF-IDF is slow
    X_train_temp = X_train_opt.drop(columns=text_cols_impute)
    X_val_temp = X_val_opt.drop(columns=text_cols_impute)
    
    model = lgb.LGBMRegressor(**lgb_params_opt)
    model.fit(X_train_temp, y_train_opt,
              eval_set=[(X_val_temp, y_val_opt)],
              callbacks=[lgb.early_stopping(50, verbose=False)])
    
    preds = model.predict(X_val_temp)
    rmse = np.sqrt(mean_squared_error(y_val_opt, preds))
    
    return rmse

# --- Run the Optuna Study ---
study = optuna.create_study(direction='minimize')
# We'll run a small number of trials. For a real competition, you might use 50-100.
study.optimize(objective, n_trials=30) 

# --- Get the Best Parameters ---
best_lgb_params = study.best_params
print("\nBest trial:")
print(f"  RMSE: {study.best_value}")
print("  Best hyperparameters: ")
for key, value in best_lgb_params.items():
    print(f"    {key}: {value}")

# --- Update our main parameter dictionaries for training ---
# We will update all three models with the tuned structural parameters
lgb_params['learning_rate'] = best_lgb_params['learning_rate']
lgb_params['num_leaves'] = best_lgb_params['num_leaves']
lgb_params['max_depth'] = best_lgb_params['max_depth']
lgb_params['subsample'] = best_lgb_params['subsample']
lgb_params['colsample_bytree'] = best_lgb_params['colsample_bytree']
lgb_params['reg_alpha'] = best_lgb_params['reg_alpha']
lgb_params['reg_lambda'] = best_lgb_params['reg_lambda']
lgb_params['min_child_samples'] = best_lgb_params['min_child_samples']

# Update XGBoost and CatBoost with equivalent parameters
xgb_params['eta'] = best_lgb_params['learning_rate']
xgb_params['max_depth'] = best_lgb_params['max_depth']
xgb_params['subsample'] = best_lgb_params['subsample']
xgb_params['colsample_bytree'] = best_lgb_params['colsample_bytree']
xgb_params['lambda'] = best_lgb_params['reg_lambda']
xgb_params['alpha'] = best_lgb_params['reg_alpha']

cb_params['learning_rate'] = best_lgb_params['learning_rate']
cb_params['depth'] = best_lgb_params['max_depth']
cb_params['subsample'] = best_lgb_params['subsample'] # Note: CatBoost has rsm (colsample_bytree) not subsample

print("\nMain model parameters updated with tuned values.")

[I 2025-07-13 21:56:45,447] A new study created in memory with name: no-name-035a4e9c-ee75-4a4c-a8e7-962b6f317d3c



--- Starting Block 2.5: Hyperparameter Tuning ---


[I 2025-07-13 21:56:51,293] Trial 0 finished with value: 120453.6896755389 and parameters: {'learning_rate': 0.021605893908811468, 'num_leaves': 108, 'max_depth': 7, 'subsample': 0.7154292845422682, 'colsample_bytree': 0.7092392616009985, 'reg_alpha': 0.0013511605675891533, 'reg_lambda': 0.0011145691214416937, 'min_child_samples': 70}. Best is trial 0 with value: 120453.6896755389.
[I 2025-07-13 21:57:00,134] Trial 1 finished with value: 114515.6536433425 and parameters: {'learning_rate': 0.07482360252673896, 'num_leaves': 294, 'max_depth': 12, 'subsample': 0.6923271598153364, 'colsample_bytree': 0.9380534510466949, 'reg_alpha': 0.0006057350075712832, 'reg_lambda': 0.0904579687601454, 'min_child_samples': 72}. Best is trial 1 with value: 114515.6536433425.
[I 2025-07-13 21:57:08,583] Trial 2 finished with value: 120934.9612432567 and parameters: {'learning_rate': 0.020174023816151868, 'num_leaves': 56, 'max_depth': 10, 'subsample': 0.9106896427115168, 'colsample_bytree': 0.743386760347


Best trial:
  RMSE: 112746.04929498266
  Best hyperparameters: 
    learning_rate: 0.08006263666752357
    num_leaves: 293
    max_depth: 12
    subsample: 0.6489751565074273
    colsample_bytree: 0.6807975842856607
    reg_alpha: 8.542840471902837
    reg_lambda: 0.043639292640908676
    min_child_samples: 15


NameError: name 'lgb_params' is not defined

In [6]:
# Block 3: Stage 1 - K-Fold Training of Mean Model Ensemble
# ==============================================================================
print("\n--- STAGE 1: K-Fold Training of Mean Model Ensemble ---")

# (The training code is now correct because the param dictionaries are global)
grade_for_stratify = pd.read_csv(DATA_PATH + 'dataset.csv')['grade']
skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)
oof_mean_preds, test_mean_preds = np.zeros(len(X)), np.zeros(len(X_test))
text_cols = ['subdivision', 'zoning', 'city', 'sale_warning', 'join_status', 'submarket']
categorical_features = text_cols + ['geo_cluster']
# Add cat_features to the params just before the loop
cb_params['cat_features'] = categorical_features

for fold, (train_idx, val_idx) in enumerate(skf.split(X, grade_for_stratify)):
    print(f"\n--- Fold {fold+1}/{N_SPLITS} ---")
    X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
    y_train, y_val = y_true.iloc[train_idx], y_true.iloc[val_idx]
    X_train_fold, X_val_fold, X_test_fold = X_train.copy(), X_val.copy(), X_test.copy()
    
    # TF-IDF Features
    for col in text_cols:
        tfidf = TfidfVectorizer(analyzer='char', ngram_range=(3, 5), max_features=150)
        X_train_fold[col], X_val_fold[col], X_test_fold[col] = (X_train_fold[col].astype(str), X_val_fold[col].astype(str), X_test_fold[col].astype(str))
        tfidf.fit(X_train_fold[col])
        train_tfidf_matrix, val_tfidf_matrix, test_tfidf_matrix = (tfidf.transform(X_train_fold[col]).toarray(), tfidf.transform(X_val_fold[col]).toarray(), tfidf.transform(X_test_fold[col]).toarray())
        num_features = train_tfidf_matrix.shape[1]
        train_tfidf, val_tfidf, test_tfidf = (pd.DataFrame(train_tfidf_matrix, index=X_train_fold.index, columns=[f'{col}_tfidf_{i}' for i in range(num_features)]), pd.DataFrame(val_tfidf_matrix, index=X_val_fold.index, columns=[f'{col}_tfidf_{i}' for i in range(num_features)]), pd.DataFrame(test_tfidf_matrix, index=X_test_fold.index, columns=[f'{col}_tfidf_{i}' for i in range(num_features)]))
        X_train_fold, X_val_fold, X_test_fold = (pd.concat([X_train_fold, train_tfidf], axis=1), pd.concat([X_val_fold, val_tfidf], axis=1), pd.concat([X_test_fold, test_tfidf], axis=1))
    X_train_fold, X_val_fold, X_test_fold = (X_train_fold.drop(columns=text_cols), X_val_fold.drop(columns=text_cols), X_test_fold.drop(columns=text_cols))
    
    # Train Models
    # XGBoost (Native API)
    dtrain, dval, dtest = (xgb.DMatrix(X_train_fold, label=y_train), xgb.DMatrix(X_val_fold, label=y_val), xgb.DMatrix(X_test_fold))
    model_xgb = xgb.train(params=xgb_params, dtrain=dtrain, num_boost_round=2500, evals=[(dval, 'eval')], early_stopping_rounds=100, verbose_eval=False)
    preds_xgb_val, preds_xgb_test = (model_xgb.predict(dval, iteration_range=(0, model_xgb.best_iteration)), model_xgb.predict(dtest, iteration_range=(0, model_xgb.best_iteration)))

    # LightGBM
    model_lgb = lgb.LGBMRegressor(**lgb_params)
    model_lgb.fit(X_train_fold, y_train, eval_set=[(X_val_fold, y_val)], callbacks=[lgb.early_stopping(100, verbose=False)])
    preds_lgb_val, preds_lgb_test = (model_lgb.predict(X_val_fold), model_lgb.predict(X_test_fold))

    # CatBoost
    X_train_cb, X_val_cb, X_test_cb = (X_train.copy(), X_val.copy(), X_test.copy())
    for col in categorical_features:
        X_train_cb[col], X_val_cb[col], X_test_cb[col] = (X_train_cb[col].astype(str), X_val_cb[col].astype(str), X_test_cb[col].astype(str))
    model_cb = cb.CatBoostRegressor(**cb_params)
    model_cb.fit(X_train_cb, y_train, eval_set=[(X_val_cb, y_val)])
    preds_cb_val, preds_cb_test = (model_cb.predict(X_val_cb), model_cb.predict(X_test_cb))

    # Ensemble Predictions
    fold_oof_preds = (preds_xgb_val + preds_lgb_val + preds_cb_val) / 3
    fold_test_preds = (preds_xgb_test + preds_lgb_test + preds_cb_test) / 3
    oof_mean_preds[val_idx] = fold_oof_preds
    test_mean_preds += fold_test_preds / N_SPLITS
    
    print(f"Fold {fold+1} OOF RMSE: {np.sqrt(mean_squared_error(y_val, fold_oof_preds)):,.2f}")
    gc.collect()

final_mean_rmse = np.sqrt(mean_squared_error(y_true, oof_mean_preds))
print(f"\n# Stage 1 (Mean) Final OOF RMSE: ${final_mean_rmse:,.2f}")


--- STAGE 1: K-Fold Training of Mean Model Ensemble ---

--- Fold 1/5 ---
Fold 1 OOF RMSE: 103,030.45

--- Fold 2/5 ---
Fold 2 OOF RMSE: 102,366.47

--- Fold 3/5 ---
Fold 3 OOF RMSE: 102,467.85

--- Fold 4/5 ---
Fold 4 OOF RMSE: 102,271.93

--- Fold 5/5 ---
Fold 5 OOF RMSE: 102,686.04

# Stage 1 (Mean) Final OOF RMSE: $102,564.90


In [7]:
# Block 4: Stage 2 - K-Fold Training of Error Model Ensemble
# ==============================================================================
print("\n--- STAGE 2: K-Fold Training of Error Model Ensemble ---")

# (This code is now correct because the param dictionaries are global)
error_target = np.abs(y_true - oof_mean_preds)
X_for_error = X.copy()
X_for_error['mean_pred_oof'] = oof_mean_preds
X_test_for_error = X_test.copy()
X_test_for_error['mean_pred_oof'] = test_mean_preds
oof_error_preds, test_error_preds = np.zeros(len(X)), np.zeros(len(X_test))

# Create separate, modified parameters for Stage 2
lgb_params_error, cb_params_error, xgb_params_error = (lgb_params.copy(), cb_params.copy(), xgb_params.copy())
lgb_params_error['n_estimators'], cb_params_error['iterations'] = 1500, 1500

for fold, (train_idx, val_idx) in enumerate(skf.split(X, grade_for_stratify)):
    print(f"\n--- Fold {fold+1}/{N_SPLITS} ---")
    X_train, X_val = X_for_error.iloc[train_idx], X_for_error.iloc[val_idx]
    y_train, y_val = error_target.iloc[train_idx], error_target.iloc[val_idx]

    # TF-IDF Features
    X_train_fold, X_val_fold, X_test_fold = X_train.copy(), X_val.copy(), X_test_for_error.copy()
    for col in text_cols:
        tfidf = TfidfVectorizer(analyzer='char', ngram_range=(3, 5), max_features=150)
        X_train_fold[col], X_val_fold[col], X_test_fold[col] = (X_train_fold[col].astype(str), X_val_fold[col].astype(str), X_test_fold[col].astype(str))
        tfidf.fit(X_train_fold[col])
        train_tfidf_matrix, val_tfidf_matrix, test_tfidf_matrix = (tfidf.transform(X_train_fold[col]).toarray(), tfidf.transform(X_val_fold[col]).toarray(), tfidf.transform(X_test_fold[col]).toarray())
        num_features = train_tfidf_matrix.shape[1]
        train_tfidf, val_tfidf, test_tfidf = (pd.DataFrame(train_tfidf_matrix, index=X_train_fold.index, columns=[f'{col}_tfidf_{i}' for i in range(num_features)]), pd.DataFrame(val_tfidf_matrix, index=X_val_fold.index, columns=[f'{col}_tfidf_{i}' for i in range(num_features)]), pd.DataFrame(test_tfidf_matrix, index=X_test_fold.index, columns=[f'{col}_tfidf_{i}' for i in range(num_features)]))
        X_train_fold, X_val_fold, X_test_fold = (pd.concat([X_train_fold, train_tfidf], axis=1), pd.concat([X_val_fold, val_tfidf], axis=1), pd.concat([X_test_fold, test_tfidf], axis=1))
    X_train_fold, X_val_fold, X_test_fold = (X_train_fold.drop(columns=text_cols), X_val_fold.drop(columns=text_cols), X_test_fold.drop(columns=text_cols))
    
    # Train Models
    # XGBoost (Native API)
    dtrain, dval, dtest = (xgb.DMatrix(X_train_fold, label=y_train), xgb.DMatrix(X_val_fold, label=y_val), xgb.DMatrix(X_test_fold))
    model_xgb = xgb.train(params=xgb_params_error, dtrain=dtrain, num_boost_round=1500, evals=[(dval, 'eval')], early_stopping_rounds=75, verbose_eval=False)
    preds_xgb_val, preds_xgb_test = (model_xgb.predict(dval, iteration_range=(0, model_xgb.best_iteration)), model_xgb.predict(dtest, iteration_range=(0, model_xgb.best_iteration)))

    # LightGBM
    model_lgb = lgb.LGBMRegressor(**lgb_params_error)
    model_lgb.fit(X_train_fold, y_train, eval_set=[(X_val_fold, y_val)], callbacks=[lgb.early_stopping(75, verbose=False)])
    preds_lgb_val, preds_lgb_test = (model_lgb.predict(X_val_fold), model_lgb.predict(X_test_fold))

    # CatBoost for Error
    X_train_cb, X_val_cb, X_test_cb = (X_train.copy(), X_val.copy(), X_test_for_error.copy())
    for col in categorical_features:
        X_train_cb[col], X_val_cb[col], X_test_cb[col] = (X_train_cb[col].astype(str), X_val_cb[col].astype(str), X_test_cb[col].astype(str))
    model_cb = cb.CatBoostRegressor(**cb_params_error)
    model_cb.fit(X_train_cb, y_train, eval_set=[(X_val_cb, y_val)])
    preds_cb_val, preds_cb_test = (model_cb.predict(X_val_cb), model_cb.predict(X_test_cb))

    # Ensemble Predictions
    fold_oof_preds = (preds_xgb_val + preds_lgb_val + preds_cb_val) / 3
    fold_test_preds = (preds_xgb_test + preds_lgb_test + preds_cb_test) / 3
    oof_error_preds[val_idx] = fold_oof_preds
    test_error_preds += fold_test_preds / N_SPLITS

    print(f"Fold {fold+1} OOF Error RMSE: {np.sqrt(mean_squared_error(y_val, fold_oof_preds)):,.2f}")
    gc.collect()

final_error_rmse = np.sqrt(mean_squared_error(error_target, oof_error_preds))
print(f"\n# Stage 2 (Error) Final OOF RMSE: ${final_error_rmse:,.2f}")


--- STAGE 2: K-Fold Training of Error Model Ensemble ---

--- Fold 1/5 ---
Fold 1 OOF Error RMSE: 66,162.64

--- Fold 2/5 ---
Fold 2 OOF Error RMSE: 64,760.24

--- Fold 3/5 ---
Fold 3 OOF Error RMSE: 65,664.18

--- Fold 4/5 ---
Fold 4 OOF Error RMSE: 65,271.85

--- Fold 5/5 ---
Fold 5 OOF Error RMSE: 65,963.57

# Stage 2 (Error) Final OOF RMSE: $65,566.42


In [8]:
# Block 5: Final Asymmetric Calibration and Submission
# ==============================================================================
print("\n--- Final Asymmetric Calibration ---")

def winkler_score(y_true, lower, upper, alpha=0.1, return_coverage=False):
    width = upper - lower
    penalty_lower = np.where(y_true < lower, (2 / alpha) * (lower - y_true), 0)
    penalty_upper = np.where(y_true > upper, (2 / alpha) * (y_true - upper), 0)
    score = width + penalty_lower + penalty_upper
    if return_coverage:
        coverage = np.mean((y_true >= lower) & (y_true <= upper))
        return np.mean(score), coverage
    return np.mean(score)

# Clip the OOF error predictions to be non-negative
oof_error_final = np.clip(oof_error_preds, 0, None)

best_a, best_b, best_metric = 2.0, 2.0, float('inf')

print("Starting grid search for optimal multipliers...")
# Define a search space based on previous results
search_space_a = np.arange(1.80, 2.21, 0.01)
search_space_b = np.arange(2.00, 2.41, 0.01)

for a in search_space_a:
    for b in search_space_b:
        low = oof_mean_preds - oof_error_final * a
        high = oof_mean_preds + oof_error_final * b
        
        metric, coverage = winkler_score(y_true, low, high, alpha=COMPETITION_ALPHA, return_coverage=True)
        
        if metric < best_metric:
            best_metric = metric
            best_a, best_b = a, b
            print(f"New best score: {best_metric:,.2f} | Coverage: {coverage:.3f} | a={best_a:.2f}, b={best_b:.2f}")

print(f"\nGrid search complete. Final OOF Score: {best_metric:,.2f}. Best multipliers: a={best_a:.2f}, b={best_b:.2f}")

# --- Create Final Submission ---
print("\nCreating final submission file...")
test_error_final = np.clip(test_error_preds, 0, None)

final_lower = test_mean_preds - test_error_final * best_a
final_upper = test_mean_preds + test_error_final * best_b
final_upper = np.maximum(final_lower, final_upper) # Ensure upper is always > lower

# Reload original test set to get the 'id' column
submission_df = pd.read_csv(DATA_PATH + 'test.csv', usecols=['id'])
submission_df['pi_lower'] = final_lower
submission_df['pi_upper'] = final_upper

submission_df.to_csv('submission_supercharged_v1.csv', index=False)
print("\n'submission_supercharged_v1.csv' created successfully!")
display(submission_df.head())


--- Final Asymmetric Calibration ---
Starting grid search for optimal multipliers...
New best score: 322,771.10 | Coverage: 0.851 | a=1.80, b=2.00
New best score: 322,485.72 | Coverage: 0.852 | a=1.80, b=2.01
New best score: 322,209.53 | Coverage: 0.853 | a=1.80, b=2.02
New best score: 321,942.04 | Coverage: 0.854 | a=1.80, b=2.03
New best score: 321,684.23 | Coverage: 0.854 | a=1.80, b=2.04
New best score: 321,435.27 | Coverage: 0.855 | a=1.80, b=2.05
New best score: 321,194.77 | Coverage: 0.856 | a=1.80, b=2.06
New best score: 320,963.65 | Coverage: 0.857 | a=1.80, b=2.07
New best score: 320,740.82 | Coverage: 0.858 | a=1.80, b=2.08
New best score: 320,528.02 | Coverage: 0.859 | a=1.80, b=2.09
New best score: 320,325.21 | Coverage: 0.859 | a=1.80, b=2.10
New best score: 320,130.78 | Coverage: 0.860 | a=1.80, b=2.11
New best score: 319,943.89 | Coverage: 0.861 | a=1.80, b=2.12
New best score: 319,765.72 | Coverage: 0.862 | a=1.80, b=2.13
New best score: 319,595.17 | Coverage: 0.862 |

,id,pi_lower,pi_upper
0,200000,821229.604746,1.089432e+06
1,200001,575707.343287,8.108854e+05
2,200002,448533.790925,6.685567e+05
3,200003,298887.233710,4.263632e+05
4,200004,325293.741879,9.079899e+05
